In [8]:
%load_ext sql
%sql duckdb:///rank_scores.duckdb

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [9]:
%%sql
Create table If Not Exists Scores (id int, score DECIMAL(3,2));
Truncate table Scores;
insert into Scores (id, score) values ('1', '3.5');
insert into Scores (id, score) values ('2', '3.65');
insert into Scores (id, score) values ('3', '4.0');
insert into Scores (id, score) values ('4', '3.85');
insert into Scores (id, score) values ('5', '4.0');
insert into Scores (id, score) values ('6', '3.65');

 * duckdb:///rank_scores.duckdb
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


Count


In [21]:
%%sql

SELECT *
FROM Scores
GROUP BY score
ORDER BY score DESC, id ASC;

 * duckdb:///rank_scores.duckdb
(_duckdb.BinderException) Binder Error: column "id" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(id)" if the exact value of "id" is not important.
[SQL: SELECT *
FROM Scores
GROUP BY score
ORDER BY score DESC, id ASC;]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [5]:
%%sql
SELECT 
    s1.score,
    (
        SELECT COUNT(DISTINCT s2.score)
        FROM Scores s2
        WHERE s2.score >= s1.score
    ) as rank
FROM Scores s1
ORDER BY s1.score DESC;

 * duckdb:///rank_scores.duckdb
Done.


score,rank


### Modern Practice Notes (Ranking vs ORDER BY)

- Use ranking (`DENSE_RANK`, `RANK`, `ROW_NUMBER`) when rank is business logic.
- Use final `ORDER BY` when you only need display order.
- Keep reusable logic in a view/CTE/subquery; do presentation sorting in the outer query.
- Always add tie-breakers for deterministic ordering (for example: `ORDER BY score DESC, id ASC`).
- In SQLAlchemy, compute ranking in a CTE/subquery, then filter/sort in the outer query.


In [ ]:
# =========================================================
# SQL Ranking + ORDER BY Guide (examples + when to use)
# =========================================================

# 1) DENSE_RANK(): 
#    - Same value => same rank, no gaps
#    - Example ranks: 1, 1, 2, 3, 3, 4
#    - Use when ties should share rank and rank numbers must be consecutive.
SELECT
  id,
  score,
  DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rank_value
FROM Scores
ORDER BY score DESC;

# 2) RANK(): 
#    - Same value => same rank, with gaps after ties
#    - Example ranks: 1, 1, 3, 4, 4, 6
#    - Use for competition-style ranking where ties skip positions.
SELECT
  id,
  score,
  RANK() OVER (ORDER BY score DESC) AS rank_value
FROM Scores
ORDER BY score DESC;

# 3) ROW_NUMBER(): 
#    - Every row gets a unique sequence number
#    - Example numbers: 1, 2, 3, 4, 5, 6
#    - Use when you need exactly one position per row (no shared ranks).
#    - Add a tie-breaker (like id) for deterministic ordering.
SELECT
  id,
  score,
  ROW_NUMBER() OVER (ORDER BY score DESC, id ASC) AS row_num
FROM Scores
ORDER BY score DESC, id ASC;

# 4) OVER (ORDER BY ...): 
#    - Defines ranking direction inside window functions
#    - DESC: higher score gets better rank.
#    - ASC : lower score gets better rank.
SELECT
  id,
  score,
  DENSE_RANK() OVER (ORDER BY score DESC) AS r
FROM Scores;

# 5) ORDER BY (final query): 
#    - Controls output display order
#    - ORDER BY at the end sorts returned rows.
#    - It does not change rank logic already defined inside OVER(...).
SELECT
  id,
  score,
  DENSE_RANK() OVER (ORDER BY score DESC) AS r
FROM Scores
ORDER BY r ASC, id ASC;

# 6) Fallback without window functions (older SQL engines)
#    - Rank = number of distinct scores >= current score
#    - Use only if window functions are unavailable.
#    - Usually slower and less readable than DENSE_RANK().
SELECT
  s1.score,
  (
    SELECT COUNT(DISTINCT s2.score)
    FROM Scores s2
    WHERE s2.score >= s1.score
  ) AS rank_value
FROM Scores s1
ORDER BY s1.score DESC;

# Quick pick for LeetCode "Rank Scores":
# DENSE_RANK() OVER (ORDER BY score DESC)


SyntaxError: unmatched ')' (1092122437.py, line 6)

# Short version: teams usually separate derived metrics from presentation order.

  1. Use ranking (row_number, rank, dense_rank) when rank itself is business logic.

  - Leaderboards
  - “Top 3 per category”
  - Dedup keep-latest row per key
  - Percentiles/cohorts

  2. Use plain ORDER BY when you only need display order.

  - Sort newest first
  - Sort by score/name/date for UI tables
  - No rank column needed

  3. View practice (PostgreSQL/startups):

  - Put reusable logic in a view/CTE (including rank if needed).
  - Do final ORDER BY in the outer query (app/API layer).
  - Avoid relying on “natural order” from a view.
  - Include tie-breakers (ORDER BY score DESC, id ASC) for deterministic results.

  4. SQLAlchemy practice:

  - Build ranking in SQLAlchemy expression layer, usually subquery/CTE.
  - Then filter/order in outer query.
  - Keep pagination separate (prefer keyset/cursor for large tables).

  Example pattern in SQLAlchemy:

  from sqlalchemy import select, func

  ranked = (
      select(
          Scores.c.id,
          Scores.c.score,
          func.dense_rank().over(order_by=Scores.c.score.desc()).label("rank"),
      ).cte("ranked")
  )

  q = (
      select(ranked.c.id, ranked.c.score, ranked.c.rank)
      .order_by(ranked.c.rank.asc(), ranked.c.id.asc())  # presentation order
  )

  Rule of thumb:

  - If product needs “position/standing,” compute rank.
  - If product only needs row order, just ORDER BY.

## Final Critique: LeetCode 178 "Rank Scores" (No `DENSE_RANK` Available)

### 1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

| Attempt | Query Shape | Correctness | Time Complexity (typical) | Space Complexity | Notes |
|---|---|---|---|---|---|
| A | `SELECT * FROM Scores GROUP BY score ...` | Invalid in DuckDB/Postgres-style SQL | N/A (fails bind) | N/A | `id` is neither grouped nor aggregated |
| B (main) | Correlated subquery with `COUNT(DISTINCT s2.score)` where `s2.score >= s1.score` | Correct dense-rank behavior | Worst-case `O(n^2)` | `O(u)` inside distinct aggregation (`u` = distinct scores in scan) | Portable fallback when window functions are unavailable |
| C (conceptual better SQL when supported) | `DENSE_RANK() OVER (ORDER BY score DESC)` | Correct | Usually `O(n log n)` from sort step | Engine-dependent (sort/window buffers) | Faster/clearer but not available under your constraint |

- Main attempt trade-off: strong portability and correctness, but weaker scalability on large/high-cardinality data.

### 2. Critique of the problem-solving approach, including progression of thought and method.

- Good pivot from invalid grouping toward rank semantics.
- Main reasoning gap: `GROUP BY` deduplicates groups; it does not automatically preserve row-level columns like `id`.
- Strong recovery: the correlated distinct-count pattern is the standard no-window fallback.
- Workflow issue: mixed SQL examples inside a Python code cell caused avoidable syntax noise and slowed iteration.

### 3. Improvements to Algorithm/ Optimal Example (include python solution code here in ``` ``` grouping braces)

```python
# Dense-rank behavior without SQL window functions.
# Useful when data is already in Python and you need deterministic ranking.

from decimal import Decimal

def rank_scores_no_dense_rank(rows):
    """
    rows: list[tuple[int, Decimal]] => (id, score)
    return: list[tuple[Decimal, int]] => (score, rank), sorted by score desc

    Complexity:
      - Time: O(n log n) due to sorting distinct scores
      - Space: O(u) for score->rank map, where u is #distinct scores
    """
    unique_scores_desc = sorted({score for _, score in rows}, reverse=True)
    rank_by_score = {score: i + 1 for i, score in enumerate(unique_scores_desc)}

    out = [(score, rank_by_score[score]) for _, score in rows]
    out.sort(key=lambda x: x[0], reverse=True)
    return out

rows = [
    (1, Decimal('3.50')),
    (2, Decimal('3.65')),
    (3, Decimal('4.00')),
    (4, Decimal('3.85')),
    (5, Decimal('4.00')),
    (6, Decimal('3.65')),
]

print(rank_scores_no_dense_rank(rows))
# [(Decimal('4.00'), 1), (Decimal('4.00'), 1), (Decimal('3.85'), 2),
#  (Decimal('3.65'), 3), (Decimal('3.65'), 3), (Decimal('3.50'), 4)]
```

### 4. Applications in real-life situations, with examples from big tech and startups (frontier tech) of the exact problem I'm solving and the solution approach. Give exact examples and usages of the generalization of the solution or pattern. Be critical and outline other algorithmic tradeoffs and when I'll use this algorithmic design/ data structure approach and when I should not.

1. **Transferable systems pattern**
   - Score-to-rank materialization: convert numeric scores into stable positions for display, qualification, or prioritization.

2. **Literal usage vs analogy**
   - Direct: analytics/BI SQL ranking queries.
   - Partial analogy: real-time leaderboards/search ranking pipelines where tie semantics and freshness constraints differ.

3. **Concrete examples**
   - BigQuery numbering functions for rank/dense rank style analytics.
   - Redis sorted sets (`ZADD`, `ZREVRANK`) for low-latency leaderboard rank retrieval.
   - MongoDB window ranking operators for rank-style aggregation.

4. **When to use and when not to use this design**
   - Use the correlated fallback when window functions are unavailable and dataset size is moderate.
   - Avoid it for high-cardinality, high-volume, low-latency paths; prefer window functions, materialized ranking tables, or online sorted data structures.

### 5. Open Questions to Challenge My Understanding (non-spoiler).

- How would you return only unique `(score, rank)` pairs while still preserving correct rank numbers?
- If score precision is inconsistent across services, what tie policy would you enforce before ranking?
- Under what distribution of scores does your correlated subquery become the bottleneck first?
- Why does `GROUP BY score` conflict with selecting `id`, and what mental model prevents this mistake?
- If ranking changes to competition rank (with gaps), how would your no-window solution change?

### 6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent.

1. **Per-game ranking (`game_id`)**
   - Learning goal intent: scoped ranking logic.
   - What changed: rank is per partition, not global.
   - Why this matters: key design and indexing must include partition columns.

2. **Streaming score updates**
   - Learning goal intent: incremental recomputation strategy.
   - What changed: scores mutate continuously.
   - Why this matters: full recomputation is expensive; pushes toward materialization/online rank stores.

3. **Top-K only API (`K=100`)**
   - Learning goal intent: latency-oriented query design.
   - What changed: full ranking not required.
   - Why this matters: can optimize heavily for top segment retrieval.

4. **Deterministic tie display (`score DESC, id ASC`)**
   - Learning goal intent: separate rank semantics from output ordering.
   - What changed: strict display order inside ties.
   - Why this matters: correctness requires both rank key and presentation key.


## SQL Syntax and Concepts: Compact Full Guide

### A. Query Skeleton (execution flow mental model)

```sql
SELECT ...
FROM ...
JOIN ... ON ...
WHERE ...
GROUP BY ...
HAVING ...
WINDOW ...
ORDER BY ...
LIMIT ... OFFSET ...;
```

- Logical order to reason about: `FROM/JOIN -> WHERE -> GROUP BY -> HAVING -> SELECT -> ORDER BY -> LIMIT`.
- `WHERE` filters rows before grouping; `HAVING` filters groups after aggregation.

### A1. What an `ident` expression looks like

In the grammar, `ident` means an identifier (name), such as a table name, column name, alias, or schema-qualified name.

Examples:

```sql
-- column identifier
salary

-- table identifier
Employee

-- qualified identifier (table.column)
Employee.salary

-- aliased table + qualified identifier
SELECT e.salary
FROM Employee AS e;

-- quoted identifier (when needed for reserved words/case)
SELECT "Salary"
FROM "Employee";
```

`ident` can appear directly as an expression in `SELECT`, `WHERE`, `ORDER BY`, and other clauses.

### B. SQL Grammar (EBNF-style, simplified)

```ebnf
query_expression ::= [with_clause] select_stmt [order_by_clause] [limit_clause] ";"

with_clause       ::= "WITH" cte_def {"," cte_def}
cte_def           ::= ident "AS" "(" query_expression ")"

select_stmt       ::= "SELECT" ["DISTINCT"] select_list
                      from_clause
                      [where_clause]
                      [group_by_clause]
                      [having_clause]
                      [window_clause]

select_list       ::= select_item {"," select_item}
select_item       ::= expr [["AS"] alias]

from_clause       ::= "FROM" table_ref {join_clause}
table_ref         ::= table_name [["AS"] alias] | "(" query_expression ")" [["AS"] alias]
join_clause       ::= join_type "JOIN" table_ref "ON" search_condition
join_type         ::= "INNER" | "LEFT" ["OUTER"] | "RIGHT" ["OUTER"] |
                      "FULL" ["OUTER"] | "CROSS"

where_clause      ::= "WHERE" search_condition
group_by_clause   ::= "GROUP" "BY" expr_list
having_clause     ::= "HAVING" search_condition
window_clause     ::= "WINDOW" window_def {"," window_def}

order_by_clause   ::= "ORDER" "BY" sort_item {"," sort_item}
sort_item         ::= expr ["ASC"|"DESC"] ["NULLS" "FIRST"|"NULLS" "LAST"]

limit_clause      ::= "LIMIT" integer ["OFFSET" integer]

expr_list         ::= expr {"," expr}
search_condition  ::= boolean_expr
expr              ::= literal | ident | function_call | "(" expr ")" | expr binary_op expr
```

- This grammar is intentionally simplified and dialect-neutral.
- Engines differ on optional features (`WINDOW`, `QUALIFY`, `FETCH FIRST`, etc.).

### C. Core Clauses

- `SELECT`: choose columns/expressions/aliases.
- `FROM`: source table/subquery/CTE.
- `JOIN`: combine tables (`INNER`, `LEFT`, `RIGHT`, `FULL`, `CROSS`).
- `WHERE`: row-level predicates.
- `GROUP BY`: collapse rows into groups for aggregates.
- `HAVING`: aggregate-aware filtering.
- `ORDER BY`: final output sort.
- `LIMIT/OFFSET`: pagination/windowing of result display.

### D. Aggregation Essentials

- Aggregates: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`.
- Rule: non-aggregated selected columns must be in `GROUP BY`.
- `COUNT(*)` counts rows; `COUNT(col)` ignores nulls; `COUNT(DISTINCT col)` counts unique non-null values.

### E. Ranking Concepts

- `ROW_NUMBER()`: unique sequence per row.
- `RANK()`: ties share rank, with gaps.
- `DENSE_RANK()`: ties share rank, no gaps.
- Without window functions, emulate dense rank using correlated distinct-count pattern.

### F. Subqueries and CTEs

- Scalar subquery: returns single value in `SELECT`/`WHERE`.
- Correlated subquery: references outer query row; can be expensive.
- CTE (`WITH`) improves readability and composability.

```sql
WITH ranked AS (
  SELECT score, COUNT(*) AS n
  FROM Scores
  GROUP BY score
)
SELECT * FROM ranked;
```

### G. NULL, Types, and Precision

- `NULL` is unknown: use `IS NULL` / `IS NOT NULL` (not `= NULL`).
- Use explicit casts for stable behavior.
- For money/scores requiring exact comparison, prefer `DECIMAL` over floating binary types.

### H. Ordering and Determinism

- SQL engines do not guarantee row order unless `ORDER BY` is provided.
- Add tie-breakers for deterministic output, e.g. `ORDER BY score DESC, id ASC`.

### I. Performance Basics

- Index columns used in joins, filters, and high-selectivity predicates.
- Avoid `SELECT *` in production paths when wide tables are costly.
- Evaluate correlated subqueries carefully on large data.
- Use `EXPLAIN` to inspect plan shape and cardinality assumptions.

### J. Transaction and Safety Basics

- `BEGIN ... COMMIT` groups writes atomically.
- `ROLLBACK` reverts uncommitted changes.
- Prefer idempotent migration scripts and explicit constraints (`PRIMARY KEY`, `UNIQUE`, `CHECK`, `FOREIGN KEY`).

### K. Interview vs Production Framing

- Interview SQL optimizes for correctness and clarity under constraints.
- Production SQL also optimizes for latency, cost, maintainability, and observability.
- Always separate rank logic from display sort and document tie semantics explicitly.


## SQL Clause Complexity Cheat Sheet (`SELECT`, `WHERE`, `JOIN`, `GROUP BY`, `HAVING`, `ORDER BY`, `PARTITION BY`)

> Complexity depends on data size, cardinality, indexes, memory, and engine execution plan. Values below are common-case heuristics.

| Clause / Operation | Typical Time Complexity | Typical Extra Space | Notes |
|---|---|---|---|
| `SELECT` (projection only) | `O(n)` | `O(1)` to `O(n)` | Reads qualifying rows; output materialization may dominate. |
| `WHERE` (no index) | `O(n)` | `O(1)` | Full scan + predicate check. |
| `WHERE` (with useful index) | `O(log n + k)` | `O(1)` | Index seek/range + return `k` matching rows. |
| `JOIN` (nested loop, no index) | `O(n*m)` | `O(1)` | Worst basic join strategy. |
| `JOIN` (index nested loop) | `O(n log m)` (roughly) | `O(1)` | Good when one side is indexed and probe side is selective. |
| `JOIN` (hash join) | `O(n + m)` average | `O(min(n,m))` | Common equi-join strategy; can spill if memory is insufficient. |
| `JOIN` (merge join) | `O(n + m)` after sort | sort cost if unsorted | If inputs already sorted on join keys, very efficient. |
| `GROUP BY` (hash aggregate) | `O(n)` average | `O(g)` | `g` = number of groups; memory pressure may cause spill. |
| `GROUP BY` (sort aggregate) | `O(n log n)` | sort buffer | Used when sorting is preferred/required by planner. |
| `HAVING` | `O(g)` after aggregation | `O(1)` to `O(g)` | Filters groups; overall cost includes full grouping work first. |
| `ORDER BY` | `O(n log n)` | `O(n)` | Full sort unless index/order-preserving plan can be used. |
| `ORDER BY ... LIMIT k` | `O(n log k)` (top-k heap) | `O(k)` | Many engines optimize top-k without full sort. |
| `PARTITION BY` in window funcs | `O(n log n)` often | `O(n)` partition/window buffers | Usually requires ordering within partitions for rank/window ops. |

### Practical interpretation by clause

- `SELECT`: cheap by itself; expensive when expressions/UDFs are heavy or wide rows are materialized.
- `WHERE`: biggest win comes from selective predicates + aligned indexes.
- `JOIN`: algorithm choice (hash/index/merge) dominates; check `EXPLAIN`.
- `GROUP BY`: near-linear with hash aggregation until memory/cardinality pressure.
- `HAVING`: never replaces `WHERE`; push row filters to `WHERE` first when possible.
- `ORDER BY`: expensive on large result sets; add `LIMIT` when product only needs top results.
- `PARTITION BY`: conceptually partitions rows, but window execution still often needs sort/order work.

### For your Rank Scores context

- Correlated fallback (`COUNT(DISTINCT s2.score) WHERE s2.score >= s1.score`) can approach `O(n^2)`.
- Window ranking (`DENSE_RANK`/`RANK`) is usually more scalable (`O(n log n)` dominated by sorting) when available.
